# Validação dos dados do case (Spark)

Notebook de **validação/reconciliação** independente do pipeline Medallion.

> **Premissa:** este notebook *não confia* na camada Gold. Ele **recalcula** as
> respostas do case a partir da Silver e do CSV bruto e reconcilia contra
> (1) as tabelas Gold, (2) os entregáveis em `output/*.csv` e (3) os valores de
> referência do case. Se os três baterem, os dados estão validados.

Rode-o no container `case-jupyter` (ou `case-spark`), **depois** de já ter
executado o pipeline (`python -m src.run_pipeline`), pois ele lê os dados Delta
das camadas `bronze`/`silver`/`gold` direto do MinIO (por caminho) e os CSVs
de `output/`.

Estratégia: **Spark** para reprocessar na escala real (4,8M linhas, Delta no
MinIO) e **pandas** apenas para ler os pequenos CSVs de `output/`.

## 0. Setup

In [1]:
import sys
from pathlib import Path

# Mesmo layout do pipeline_medallion.ipynb: torna o pacote `src` importável.
sys.path.insert(0, "/app")

import pandas as pd
from pyspark.sql import functions as F
from src.config import get_spark, csv_path, lake_root

spark = get_spark("case-ans-validacao")
print("Spark :", spark.version)
print("CSV   :", csv_path())
print("Lake  :", lake_root())

:: loading settings :: url = jar:file:/usr/local/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-b549312b-8e90-4b54-9563-2c333625fb75;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 205ms :: artifacts dl 8ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 f

Spark : 3.5.1
CSV   : /app/data/pda-024-icb-SP-2025_08.csv
Lake  : s3a://lakehouse


Spark : 3.5.1
CSV   : /app/data/pda-024-icb-SP-2025_08.csv
Lake  : s3a://lakehouse


In [2]:
# Coletor de resultados: cada verificação registra um PASS/FAIL aqui.
# Função usada no final do notebook
RESULTS = []

def check(nome, ok, detalhe=""):
    """Registra e imprime o resultado de uma verificação."""
    ok = bool(ok)
    RESULTS.append((nome, ok, detalhe))
    marca = "✅ PASS" if ok else "❌ FAIL"
    print(f"{marca} | {nome}" + (f"  ->  {detalhe}" if detalhe else ""))
    return ok

## V1 — Integridade de volume

O CSV bruto, a Bronze e a Silver devem ter exatamente o mesmo número de linhas:
a ingestão copia 1:1 e a tipagem da Silver não pode descartar registros.

In [4]:
# Tabelas do lakehouse lidas DIRETO do MinIO (Delta por caminho), e nao pelo
# catalogo Hive. Motivo: o metastore Derby (/app/metastore_db) nao e um volume
# compartilhado, entao o catalogo do container que rodou o pipeline (case-spark)
# nao e visivel aqui (case-jupyter). Os dados Delta, porem, vivem no MinIO
# (compartilhado) -- le-los por caminho torna a validacao independente do
# container onde o pipeline rodou.
LAKE = lake_root()  # s3a://lakehouse

def delta(path):
    return spark.read.format("delta").load(f"{LAKE}/{path}")

bronze  = delta("bronze/beneficiarios")
silver  = delta("silver/beneficiarios")
g_oper  = delta("gold/beneficiarios_por_operadora")
g_faixa = delta("gold/beneficiarios_por_faixa_etaria")
g_muni  = delta("gold/beneficiarios_por_municipio")

# Relê o CSV bruto com AS MESMAS opções do Bronze (independente do Delta).
raw = (spark.read
       .option("header", "true").option("sep", ";").option("quote", '"')
       .option("encoding", "UTF-8").option("mode", "PERMISSIVE")
       .csv(csv_path()))
n_raw = raw.count()

check("Bronze preserva todas as linhas do CSV bruto", n_bronze == n_raw,
      f"csv={n_raw:,} bronze={n_bronze:,}")
check("Silver e Bronze tem a mesma contagem (tipagem 1:1)", n_silver == n_bronze,
      f"bronze={n_bronze:,} silver={n_silver:,}")

✅ PASS | Bronze preserva todas as linhas do CSV bruto  ->  csv=4,830,707 bronze=4,830,707
✅ PASS | Silver e Bronze tem a mesma contagem (tipagem 1:1)  ->  bronze=4,830,707 silver=4,830,707


True

## V2 — Tipagem sem perda

O `CAST(... AS INT)` das colunas de quantidade não pode ter transformado valores
válidos em `NULL`. Comparamos os nulos na Silver com os campos realmente
vazios/nulos na origem (Bronze): a Silver não deve ter *mais* nulos que a origem.

In [5]:
def nulos_silver(col):
    return silver.filter(F.col(col).isNull()).count()

def vazios_bronze(col):
    return bronze.filter(F.col(col).isNull() | (F.trim(F.col(col)) == "")).count()

pares = [
    ("qt_beneficiario_ativo",     "QT_BENEFICIARIO_ATIVO"),
    ("qt_beneficiario_aderido",   "QT_BENEFICIARIO_ADERIDO"),
    ("qt_beneficiario_cancelado", "QT_BENEFICIARIO_CANCELADO"),
]
for col_s, col_b in pares:
    ns, nb = nulos_silver(col_s), vazios_bronze(col_b)
    check(f"CAST INT nao gerou NULL indevido em {col_s}", ns <= nb,
          f"nulos_silver={ns} vazios_origem={nb}")

check("dt_carga convertida para DATE (ha valores nao-nulos)",
      silver.filter(F.col("dt_carga").isNotNull()).count() > 0)

✅ PASS | CAST INT nao gerou NULL indevido em qt_beneficiario_ativo  ->  nulos_silver=0 vazios_origem=0
✅ PASS | CAST INT nao gerou NULL indevido em qt_beneficiario_aderido  ->  nulos_silver=0 vazios_origem=0
✅ PASS | CAST INT nao gerou NULL indevido em qt_beneficiario_cancelado  ->  nulos_silver=0 vazios_origem=0
✅ PASS | dt_carga convertida para DATE (ha valores nao-nulos)


True

True

## V3 — Mascaramento (LGPD)

Regra da Silver: `nr_cnpj_masc = CONCAT(SUBSTRING(NR_CNPJ,1,8),'******')`.
Todo registro deve terminar em 6 asteriscos e **nenhum** pode expor mais que a
raiz de 8 dígitos (CNPJ completo jamais vazado).

In [6]:
mascarados = silver.filter(F.col("nr_cnpj_masc").rlike(r"\*{6}$")).count()
check("100% dos CNPJ terminam em 6 asteriscos", mascarados == n_silver,
      f"{mascarados:,}/{n_silver:,}")

# Conta apenas os digitos visiveis: nao pode passar da raiz (8).
digitos = silver.withColumn(
    "nd", F.length(F.regexp_replace("nr_cnpj_masc", r"[^0-9]", "")))
max_dig = digitos.agg(F.max("nd")).first()[0]
check("Nenhum CNPJ expoe mais que a raiz (<=8 digitos)", max_dig <= 8,
      f"max_digitos_visiveis={max_dig}")

✅ PASS | 100% dos CNPJ terminam em 6 asteriscos  ->  4,830,707/4,830,707
✅ PASS | Nenhum CNPJ expoe mais que a raiz (<=8 digitos)  ->  max_digitos_visiveis=8


True

## V4 — Sanidade de valores

Quantidade de beneficiários ativos não pode ser negativa.

In [7]:
neg = silver.filter(F.col("qt_beneficiario_ativo") < 0).count()
check("Sem quantidade de ativos negativa na Silver", neg == 0, f"negativos={neg}")

✅ PASS | Sem quantidade de ativos negativa na Silver  ->  negativos=0


True

## V5 — Gold == recomputo independente da Silver

Recalculamos as 3 agregações **direto da Silver** e comparamos com as tabelas
Gold. Zero divergências = a Gold agregou corretamente.

In [8]:
r_oper  = (silver.groupBy("cd_operadora")
           .agg(F.sum("qt_beneficiario_ativo").alias("qt")))
r_faixa = (silver.groupBy("de_faixa_etaria")
           .agg(F.sum("qt_beneficiario_ativo").alias("qt")))
r_muni  = (silver.groupBy("cd_municipio")
           .agg(F.sum("qt_beneficiario_ativo").alias("qt")))

def divergencias(df_recalc, df_gold, key):
    """Nº de chaves com soma divergente ou presente em só um dos lados."""
    j = df_recalc.alias("r").join(
        df_gold.select(key, F.col("qt_beneficiarios_ativos").alias("qg")).alias("g"),
        on=key, how="full_outer")
    return j.filter((F.col("qt") != F.col("qg")) |
                    F.col("qt").isNull() | F.col("qg").isNull()).count()

check("Gold operadora == recomputo da Silver",
      divergencias(r_oper, g_oper, "cd_operadora") == 0)
check("Gold faixa etaria == recomputo da Silver",
      divergencias(r_faixa, g_faixa, "de_faixa_etaria") == 0)
check("Gold municipio == recomputo da Silver",
      divergencias(r_muni, g_muni, "cd_municipio") == 0)

✅ PASS | Gold operadora == recomputo da Silver
✅ PASS | Gold faixa etaria == recomputo da Silver
✅ PASS | Gold municipio == recomputo da Silver


True

## V6 — Entregáveis (`output/*.csv`) == Gold

Os CSVs consumidos pelo dashboard devem bater, valor a valor, com a Gold.

In [9]:
OUT = Path("/app/output")

# (a) Top 5 operadoras
a_out  = pd.read_csv(OUT / "a_top5_operadoras.csv")
a_gold = (g_oper.orderBy(F.desc("qt_beneficiarios_ativos")).limit(5).toPandas())
check("output/a_top5_operadoras bate com a Gold",
      list(a_out["qt_beneficiarios_ativos"]) == list(a_gold["qt_beneficiarios_ativos"]),
      f"csv={list(a_out['qt_beneficiarios_ativos'])}")

# (b) Faixa etaria lider
b_out  = pd.read_csv(OUT / "b_faixa_etaria_top.csv")
b_gold = g_faixa.orderBy(F.desc("qt_beneficiarios_ativos")).limit(1).toPandas()
check("output/b_faixa_etaria_top bate com a Gold",
      int(b_out["qt_beneficiarios_ativos"][0]) == int(b_gold["qt_beneficiarios_ativos"][0])
      and b_out["de_faixa_etaria"][0] == b_gold["de_faixa_etaria"][0],
      f"{b_out['de_faixa_etaria'][0]} = {int(b_out['qt_beneficiarios_ativos'][0]):,}")

# (c) Municipios (comparacao completa, valor a valor)
c_out  = pd.read_csv(OUT / "c_beneficiarios_por_municipio.csv")
c_gold = g_muni.orderBy(F.desc("qt_beneficiarios_ativos")).toPandas()
check("output/c_municipios tem mesma contagem de linhas que a Gold",
      len(c_out) == len(c_gold), f"csv={len(c_out)} gold={len(c_gold)}")
check("output/c_municipios bate valor a valor com a Gold",
      list(c_out["qt_beneficiarios_ativos"]) == list(c_gold["qt_beneficiarios_ativos"]))

✅ PASS | output/a_top5_operadoras bate com a Gold  ->  csv=[4324507, 2844728, 2671950, 2111707, 1391580]
✅ PASS | output/b_faixa_etaria_top bate com a Gold  ->  40 a 44 anos = 3,100,823
✅ PASS | output/c_municipios tem mesma contagem de linhas que a Gold  ->  csv=646 gold=646
✅ PASS | output/c_municipios bate valor a valor com a Gold


True

## V7 — Consistência cruzada de somas

O total de beneficiários ativos deve ser **idêntico** na Silver e nas três
tabelas Gold — todas agregam o mesmo universo por dimensões diferentes.

In [10]:
t_silver = silver.agg(F.sum("qt_beneficiario_ativo")).first()[0]
t_oper   = g_oper.agg(F.sum("qt_beneficiarios_ativos")).first()[0]
t_faixa  = g_faixa.agg(F.sum("qt_beneficiarios_ativos")).first()[0]
t_muni   = g_muni.agg(F.sum("qt_beneficiarios_ativos")).first()[0]

check("Soma total de ativos identica em Silver e nas 3 Golds",
      t_silver == t_oper == t_faixa == t_muni,
      f"silver={t_silver:,} oper={t_oper:,} faixa={t_faixa:,} muni={t_muni:,}")

✅ PASS | Soma total de ativos identica em Silver e nas 3 Golds  ->  silver=29,917,951 oper=29,917,951 faixa=29,917,951 muni=29,917,951


True

## V8 — Conferência contra os valores de referência do case

Resultados esperados para a competência **2025-08** (documentados no `CLAUDE.md`).

In [11]:
REF = {
    "top_operadora_nome": "NOTRE DAME INTERMEDICA SAUDE S.A.",
    "top_operadora_qt":   4324507,
    "top_faixa":          "40 a 44 anos",
    "top_faixa_qt":       3100823,
    "top_muni":           "Sao Paulo",
    "top_muni_qt":        9833635,
    "qt_municipios":      646,
}

def _norm(s):
    """Normaliza acentos para comparar nomes sem depender de encoding."""
    import unicodedata
    return "".join(c for c in unicodedata.normalize("NFKD", s)
                   if not unicodedata.combining(c))

top_o  = g_oper.orderBy(F.desc("qt_beneficiarios_ativos")).first()
top_f  = g_faixa.orderBy(F.desc("qt_beneficiarios_ativos")).first()
top_m  = g_muni.orderBy(F.desc("qt_beneficiarios_ativos")).first()
n_muni = g_muni.count()

check("(a) Top operadora == referencia do case",
      _norm(top_o["nm_razao_social"]) == REF["top_operadora_nome"]
      and top_o["qt_beneficiarios_ativos"] == REF["top_operadora_qt"],
      f"{top_o['nm_razao_social']} = {top_o['qt_beneficiarios_ativos']:,}")
check("(b) Faixa etaria lider == referencia do case",
      top_f["de_faixa_etaria"] == REF["top_faixa"]
      and top_f["qt_beneficiarios_ativos"] == REF["top_faixa_qt"],
      f"{top_f['de_faixa_etaria']} = {top_f['qt_beneficiarios_ativos']:,}")
check("(c) Municipio lider == referencia do case",
      _norm(top_m["nm_municipio"]) == REF["top_muni"]
      and top_m["qt_beneficiarios_ativos"] == REF["top_muni_qt"],
      f"{top_m['nm_municipio']} = {top_m['qt_beneficiarios_ativos']:,}")
check("(c) Quantidade de municipios == referencia do case",
      n_muni == REF["qt_municipios"], f"municipios={n_muni}")

✅ PASS | (a) Top operadora == referencia do case  ->  NOTRE DAME INTERMÉDICA SAÚDE S.A. = 4,324,507
✅ PASS | (b) Faixa etaria lider == referencia do case  ->  40 a 44 anos = 3,100,823
✅ PASS | (c) Municipio lider == referencia do case  ->  São Paulo = 9,833,635
✅ PASS | (c) Quantidade de municipios == referencia do case  ->  municipios=646


True

## Resumo

Consolida todos os PASS/FAIL. O `assert` final falha o notebook se algo quebrar.

In [11]:
resumo = pd.DataFrame(RESULTS, columns=["verificacao", "passou", "detalhe"])
n_ok, n_tot = int(resumo["passou"].sum()), len(resumo)
print(f"=== RESUMO: {n_ok}/{n_tot} verificacoes passaram ===\n")
display(resumo)

assert n_ok == n_tot, f"{n_tot - n_ok} verificacao(oes) falharam -- dados NAO validados."
print("\n\U0001f389 Todos os dados foram validados com sucesso.")

=== RESUMO: 17/17 verificacoes passaram ===



,verificacao,passou,detalhe
0,Bronze preserva todas as linhas do CSV bruto,True,"csv=4,830,707 bronze=4,830,707"
1,Silver e Bronze tem a mesma contagem (tipagem ...,True,"bronze=4,830,707 silver=4,830,707"
2,CAST INT nao gerou NULL indevido em qt_benefic...,True,nulos_silver=0 vazios_origem=0
3,CAST INT nao gerou NULL indevido em qt_benefic...,True,nulos_silver=0 vazios_origem=0
4,CAST INT nao gerou NULL indevido em qt_benefic...,True,nulos_silver=0 vazios_origem=0
5,dt_carga convertida para DATE (ha valores nao-...,True,
6,100% dos CNPJ terminam em 6 asteriscos,True,"4,830,707/4,830,707"
7,Nenhum CNPJ expoe mais que a raiz (<=8 digitos),True,max_digitos_visiveis=8
8,Sem quantidade de ativos negativa na Silver,True,negativos=0
9,Gold operadora == recomputo da Silver,True,



🎉 Todos os dados foram validados com sucesso.
